In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path as _P
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score)
import sklearn
print('sklearn:', sklearn.__version__)
REPO_ROOT = _P.cwd()
if not (REPO_ROOT / 'data' / 'processed' / 'customer_behavior_features.csv').exists():
    REPO_ROOT = _P.cwd().parent
FEAT_CSV = REPO_ROOT / 'data' / 'processed' / 'customer_behavior_features.csv'
PIPE_PATH = REPO_ROOT / 'models' / 'customer_behavior_feature_pipeline.joblib'
DICT_CSV = REPO_ROOT / 'reports' / 'customer_behavior_feature_dictionary.csv'
OUT_DIR = REPO_ROOT / 'reports' / 'customer_behavior_clustering'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print('REPO_ROOT:', REPO_ROOT)

## 1. Load Customer Behaviour Features

One row = one customer. Identifiers kept aside for output; never re-aggregate unless a check forces it.

In [ ]:
customer_features = pd.read_csv(FEAT_CSV)
print('shape:', customer_features.shape)
print('columns:', list(customer_features.columns))
print(customer_features.dtypes.to_string())
print('customers:', len(customer_features), '| missing cells:', int(customer_features.isna().sum().sum()))
assert customer_features['customer_id'].is_unique, 'Duplicate customers - grain violated'
print('grain OK: 1 row = 1 customer')

## 2. Dataset Verification

Uniqueness, identifier presence, no supervised target leakage into the frame.

In [ ]:
assert customer_features['customer_id'].is_unique
assert 'customer_name' in customer_features.columns
assert 'is_late' not in customer_features.columns, 'Supervised label present - must not train on it'
print('customers:', len(customer_features), '| no is_late column: PASS')
if DICT_CSV.exists():
    dd = pd.read_csv(DICT_CSV); print('dictionary rows:', len(dd))

## 3. Clustering Feature Selection

Explicit `CLUSTER_FEATURES` from the engineering pipeline; forbidden identifiers rejected loudly.

In [ ]:
bundle0 = joblib.load(PIPE_PATH)
CLUSTER_FEATURES = list(bundle0['features'])
print(f'CLUSTER_FEATURES ({len(CLUSTER_FEATURES)}):'); print(CLUSTER_FEATURES)
forbidden = {'customer_id', 'customer_name', 'invoice_id'}
hit = forbidden.intersection(CLUSTER_FEATURES)
print('forbidden hit:', hit if hit else 'none - PASS')
assert not hit, f'Forbidden features in CLUSTER_FEATURES: {hit}. STOPPING.'
assert 'is_late' not in CLUSTER_FEATURES and 'delay_days' not in CLUSTER_FEATURES

## 4. Apply Feature Preprocessing

Saved pipeline (median imputation + RobustScaler) loaded and tested; subsets never refit.

In [ ]:
bundle = joblib.load(PIPE_PATH)
feature_pipeline = bundle['pipeline']
print('pipeline:', feature_pipeline)
print('reference_date:', bundle.get('reference_date'), '| observation:', bundle.get('observation_period'))
X = customer_features[CLUSTER_FEATURES].copy()
for c in CLUSTER_FEATURES: X[c] = pd.to_numeric(X[c], errors='coerce')
print('raw X NaN cells (structural no-history):', int(X.isna().sum().sum()))
assert not np.isinf(X.to_numpy(dtype=float, na_value=np.nan)).any()
assert set(['customer_id','customer_name']).isdisjoint(X.columns)
print('matrix checks PASSED (identifiers excluded, no inf, no string columns)')

## 5. Feature Distribution Analysis

Pre-scaling spread check: medians/IQRs and NaN share per feature (why RobustScaler is appropriate).

In [ ]:
desc = X[CLUSTER_FEATURES].describe(percentiles=[0.5]).T[['count','mean','50%','std','min','max']]
desc['nan_share'] = 1 - desc['count'] / len(X)
print(desc.round(3).to_string())
print('zero-variance:', [c for c in CLUSTER_FEATURES if X[c].nunique(dropna=True) <= 1] or 'none')

In [ ]:
X_arr = feature_pipeline.transform(X)
names = list(feature_pipeline.get_feature_names_out()) if hasattr(feature_pipeline, 'get_feature_names_out') else CLUSTER_FEATURES
X_cluster = pd.DataFrame(X_arr, columns=names, index=X.index)
assert np.isfinite(X_cluster.to_numpy(dtype=float)).all(), 'Non-finite after preprocessing'
med = np.median(X_cluster.to_numpy(dtype=float), axis=0)
print('post-scaling median abs (should be ~0 for RobustScaler):', round(float(np.median(np.abs(med))), 4))
print('X_cluster:', X_cluster.shape)
Xn = X_cluster.to_numpy(dtype=float)

## 6. Baseline Clustering

Reference only: `KMeans(n_clusters=2, random_state=42, n_init=10)`. Silhouette / Calinski-Harabasz / Davies-Bouldin / Inertia. Never the final model.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
def internal_metrics(Xm, labels):
    lab = np.asarray(labels); uniq = sorted(set(lab) - {-1})
    n_c = len(uniq)
    if n_c < 2 or n_c >= len(lab):
        return {'n_clusters': n_c, 'silhouette': np.nan, 'ch': np.nan, 'db': np.nan}
    mask = lab != -1
    if mask.sum() < 3 or len(set(lab[mask])) < 2:
        return {'n_clusters': n_c, 'silhouette': np.nan, 'ch': np.nan, 'db': np.nan}
    return {'n_clusters': n_c, 'silhouette': float(silhouette_score(Xm[mask], lab[mask])),
            'ch': float(calinski_harabasz_score(Xm[mask], lab[mask])), 'db': float(davies_bouldin_score(Xm[mask], lab[mask]))}
base = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
base_lab = base.fit_predict(Xn)
bm = internal_metrics(Xn, base_lab); bm['inertia'] = float(base.inertia_)
print('Baseline KMeans(2):', {k: round(v, 4) if v == v else v for k, v in bm.items()})
print('sizes:', np.bincount(base_lab))

## 7. K-Means Clustering

K=2..10 (`random_state=42, n_init=10`). Inertia/Silhouette/CH/DB per K; elbow, silhouette, DB and CH plots saved.

In [ ]:
KM_ROWS = []
KM_LABELS = {}
for k in range(2, 11):
    m = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = m.fit_predict(Xn)
    r = internal_metrics(Xn, lab); r.update({'K': k, 'inertia': float(m.inertia_), 'min_size': int(np.bincount(lab).min()), 'max_size': int(np.bincount(lab).max())})
    KM_ROWS.append(r); KM_LABELS[k] = lab
km_df = pd.DataFrame(KM_ROWS)
print(km_df.round(4).to_string(index=False))
def _sv(nm, x, y, xl, yl, inv=False):
    plt.figure(figsize=(7,4)); plt.plot(x, y, marker='o')
    plt.xlabel(xl); plt.ylabel(yl); plt.title(nm); plt.grid(alpha=0.3)
    p = OUT_DIR / nm; plt.tight_layout(); plt.savefig(p, dpi=120); plt.show(); print('saved:', p)
_sv('kmeans_elbow.png', km_df['K'], km_df['inertia'], 'K', 'Inertia')
_sv('kmeans_silhouette.png', km_df['K'], km_df['silhouette'], 'K', 'Silhouette (higher=better)')
_sv('kmeans_db.png', km_df['K'], km_df['db'], 'K', 'Davies-Bouldin (lower=better)')
_sv('kmeans_ch.png', km_df['K'], km_df['ch'], 'K', 'Calinski-Harabasz (higher=better)')

## 8. Cluster Number Analysis

K is never chosen by highest silhouette alone: K=2 scores ~0.99 only by isolating ONE extreme mega-customer (degenerate, size-1).
Reading metrics together — silhouette elbow after the outlier splits, DB minimum region, CH growth, and actionable sizes —
the evidence concentrates around K=5..8; the final pick is documented in §13 after algorithm comparison.

In [ ]:
print(km_df[['K','silhouette','ch','db','inertia','min_size','max_size']].round(4).to_string(index=False))
print('Note: min_size 1-2 at low K = single mega-customers peeled off (see profiling); not business segments.')

## 9. Hierarchical / Agglomerative Clustering

ward/complete/average over small K ranges (cheap at n=1425); linkage-compatible metrics only (euclidean).

In [ ]:
AG_ROWS = []
AG_LABELS = {}
for linkage, ks in [('ward', range(2, 11)), ('complete', range(2, 7)), ('average', range(2, 7))]:
    for k in ks:
        lab = AgglomerativeClustering(n_clusters=k, linkage=linkage).fit_predict(Xn)
        r = internal_metrics(Xn, lab); r.update({'linkage': linkage, 'K': k, 'min_size': int(np.bincount(lab).min()), 'max_size': int(np.bincount(lab).max())})
        AG_ROWS.append(r); AG_LABELS[(linkage, k)] = lab
ag_df = pd.DataFrame(AG_ROWS)
print(ag_df.round(4).to_string(index=False))

## 10. DBSCAN Clustering

Light eps/min_samples grid. Noise-aware metrics (computed on non-noise points, >=2 clusters); 0/1-cluster or almost-all-noise configs flagged invalid, never selected on a bare number.

In [ ]:
DB_ROWS = []
DB_LABELS = {}
for eps in [0.5, 1.0, 2.0, 3.0]:
    for ms in [5, 10]:
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xn)
        n_noise = int((lab == -1).sum()); n_cl = len(set(lab) - {-1})
        r = internal_metrics(Xn, lab)
        valid = (n_cl >= 2) and (n_noise / len(lab) < 0.90)
        DB_ROWS.append({'eps': eps, 'min_samples': ms, 'n_clusters': n_cl, 'noise': n_noise,
                        'noise_pct': round(100 * n_noise / len(lab), 2), 'silhouette': r['silhouette'],
                        'db': r['db'], 'valid': valid})
        DB_LABELS[(eps, ms)] = lab
db_df = pd.DataFrame(DB_ROWS)
print(db_df.round(4).to_string(index=False))
print('valid DBSCAN configs:', int(db_df['valid'].sum()), '(high-noise/outlier-heavy data Expected to suit K-Means better)')

## 11. Cluster Evaluation

Degeneracy screen: single-cluster, singleton-per-customer, and >=90%-noise outcomes are invalid by rule (`n_clusters >= 2` minimum). Sizes per candidate.

In [ ]:
print('K-Means sizes:')
for k in range(2, 11): print(f'  K={k}:', sorted(np.bincount(KM_LABELS[k]).tolist()))
print('Agglomerative (ward) sizes:')
for k in range(2, 11): print(f'  K={k}:', sorted(np.bincount(AG_LABELS[("ward", k)]).tolist()))
print('Tiny-cluster verdict: size-1/2 groups = extreme mega-customers (niche outlier behaviour, profiled in §14) — reported, not silently deleted.')

## 12. Model / Configuration Comparison

Full cross-algorithm table saved to `model_comparison.csv`.

In [ ]:
rows = []
for r in KM_ROWS:
    rows.append({'Algorithm': 'KMeans', 'Parameters': f"K={r['K']}", 'Number of Clusters': r['n_clusters'], 'Silhouette Score': r['silhouette'], 'Calinski-Harabasz Score': r['ch'], 'Davies-Bouldin Score': r['db'], 'Noise Percentage': 0.0, 'Minimum Cluster Size': r['min_size'], 'Maximum Cluster Size': r['max_size']})
for r in AG_ROWS:
    rows.append({'Algorithm': f"Agglomerative-{r['linkage']}", 'Parameters': f"K={r['K']}", 'Number of Clusters': r['n_clusters'], 'Silhouette Score': r['silhouette'], 'Calinski-Harabasz Score': r['ch'], 'Davies-Bouldin Score': r['db'], 'Noise Percentage': 0.0, 'Minimum Cluster Size': r['min_size'], 'Maximum Cluster Size': r['max_size']})
for r in DB_ROWS:
    rows.append({'Algorithm': 'DBSCAN', 'Parameters': f"eps={r['eps']},min_samples={r['min_samples']}", 'Number of Clusters': r['n_clusters'], 'Silhouette Score': r['silhouette'], 'Calinski-Harabasz Score': np.nan, 'Davies-Bouldin Score': r['db'], 'Noise Percentage': r['noise_pct'], 'Minimum Cluster Size': '', 'Maximum Cluster Size': ''})
comp_df = pd.DataFrame(rows)
comp_df.to_csv(OUT_DIR / 'model_comparison.csv', index=False)
print(comp_df.round(4).to_string(index=False)); print('saved: model_comparison.csv')

## 13. Select Final Clustering Configuration

Documented rules (no weighted overall score): valid (>=2 clusters, DBSCAN noise<90%) -> actionable (>=3 segments with >=10 customers, so segments are interpretable business groups, micro-outlier niches aside) -> lowest Davies-Bouldin, tie-break higher silhouette.
Applied to K-Means and Agglomerative; the algorithm winner needs lower DB plus better stability/interpretability.

In [ ]:
def actionable(sizes, floor=10, need=3): return sum(s >= floor for s in sizes) >= need
km_cand = [r for r in KM_ROWS if actionable(np.bincount(KM_LABELS[r['K']]))]
ag_cand = [r for r in AG_ROWS if actionable(np.bincount(AG_LABELS[(r['linkage'], r['K'])]))]
km_best = min(km_cand, key=lambda r: (r['db'], -r['silhouette']))
ag_best = min(ag_cand, key=lambda r: (r['db'], -r['silhouette'])) if ag_cand else None
print('K-Means candidates (actionable):', [(r['K'], round(r['db'],4)) for r in km_cand])
print('K-Means best:', km_best['K'], 'DB=', round(km_best['db'],4), 'Sil=', round(km_best['silhouette'],4))
if ag_best: print('Agg best:', ag_best['linkage'], ag_best['K'], 'DB=', round(ag_best['db'],4), 'Sil=', round(ag_best['silhouette'],4))
if ag_best is not None and ag_best['db'] < km_best['db'] - 0.05:
    FINAL_ALGORITHM, FINAL_PARAMETERS, FINAL_N_CLUSTERS = f"Agglomerative-{ag_best['linkage']}", {'linkage': ag_best['linkage'], 'n_clusters': ag_best['K']}, ag_best['K']
    final_labels = AG_LABELS[(ag_best['linkage'], ag_best['K'])]
else:
    FINAL_ALGORITHM, FINAL_PARAMETERS, FINAL_N_CLUSTERS = 'KMeans', {'n_clusters': km_best['K'], 'random_state': RANDOM_STATE, 'n_init': 10}, km_best['K']
    final_model = KMeans(**FINAL_PARAMETERS); final_labels = final_model.fit_predict(Xn)
print('FINAL:', FINAL_ALGORITHM, FINAL_PARAMETERS, 'K =', FINAL_N_CLUSTERS)
print('sizes:', sorted(np.bincount(final_labels).tolist()))

## 14. Customer Segment Profiling

Counts/percentages plus median/mean of behavioural features per cluster (medians for skewed financials). Neutral descriptions first — no Good/Bad labels.

In [ ]:
customer_clusters = customer_features.copy()
customer_clusters['cluster_id'] = final_labels
assert len(customer_clusters) == len(customer_features) and customer_clusters['cluster_id'].notna().all()
PROF = ['invoice_count','total_invoice_amount','avg_invoice_amount','avg_payment_delay','median_payment_delay','std_payment_delay','late_payment_ratio','paid_invoice_count','open_invoice_count','days_since_last_invoice','customer_lifetime_days','recent_avg_delay','recent_late_ratio']
prof = customer_clusters.groupby('cluster_id').agg(customer_count=('customer_id','size'), **{f'median_{c}': (c,'median') for c in PROF}, **{f'mean_{c}': (c,'mean') for c in ['late_payment_ratio','avg_payment_delay','avg_invoice_amount','invoice_count']})
prof['customer_percentage'] = round(100 * prof['customer_count'] / len(customer_clusters), 2)
prof = prof.reset_index().sort_values('cluster_id')
print(prof.round(3).to_string(index=False))

In [ ]:
prof.to_csv(OUT_DIR / 'cluster_profiles.csv', index=False)
print('saved: cluster_profiles.csv', prof.shape)
print('Micro-cluster check: sizes <10 are extreme mega-customer niches (see max invoice_count/total per cluster):')
print(customer_clusters.groupby('cluster_id')[['invoice_count','total_invoice_amount','late_payment_ratio']].max().round(2).to_string())

## 15. Cluster Visualization

PCA(2) scatter of the clustered matrix (visualization only — clustering stays in the original scaled space). UMAP only if installed (it is not -> graceful skip).

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
P2 = pca.fit_transform(Xn)
print('PCA explained variance:', np.round(pca.explained_variance_ratio_, 4))
plt.figure(figsize=(8,6))
for k in sorted(set(final_labels)):
    m = final_labels == k; plt.scatter(P2[m,0], P2[m,1], s=10, alpha=0.7, label=f'cluster {k} (n={(final_labels==k).sum()})')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'PCA 2D — {FINAL_ALGORITHM} K={FINAL_N_CLUSTERS} (visualization only)'); plt.legend(markerscale=2)
plt.tight_layout(); plt.savefig(OUT_DIR / 'pca_clusters.png', dpi=120); plt.show(); print('saved: pca_clusters.png')
try:
    import umap
    e = umap.UMAP(n_components=2, random_state=RANDOM_STATE).fit_transform(Xn)
    plt.figure(figsize=(8,6))
    for k in sorted(set(final_labels)):
        m = final_labels == k; plt.scatter(e[m,0], e[m,1], s=10, alpha=0.7, label=f'cluster {k}')
    plt.title('UMAP 2D (visualization only, not clustering features)'); plt.legend(markerscale=2)
    plt.tight_layout(); plt.savefig(OUT_DIR / 'umap_clusters.png', dpi=120); plt.show(); print('saved: umap_clusters.png')
except Exception as ex:
    print('UMAP skipped (not installed):', type(ex).__name__, '| UMAP visualization != clustering algorithm')

## 16. Business Interpretation

Descriptive profiles per cluster (observed distributions only; no causal or Good/Bad claims). FlowCred use: collections prioritization, segmentation, AR monitoring, differentiated risk strategy — descriptive segmentation, never automatic credit decisions.

In [ ]:
HEATF = ['invoice_count','avg_invoice_amount','avg_payment_delay','late_payment_ratio','std_payment_delay','recent_avg_delay','recent_late_ratio','days_since_last_invoice']
med = customer_clusters.groupby('cluster_id')[HEATF].median()
z = (med - med.mean()) / med.std().replace(0, 1)
plt.figure(figsize=(10, max(3, 0.8 * len(med) + 2))); plt.imshow(z.values, cmap='coolwarm', vmin=-2, vmax=2, aspect='auto'); plt.colorbar(label='z across clusters')
plt.xticks(range(len(HEATF)), HEATF, rotation=30, ha='right'); plt.yticks(range(len(med)), [f'cluster {i}' for i in med.index])
plt.title('Cluster-profile heatmap (median, standardized)'); plt.tight_layout()
plt.savefig(OUT_DIR / 'cluster_profile_heatmap.png', dpi=120); plt.show(); print('saved: cluster_profile_heatmap.png')
sz = np.bincount(final_labels)
plt.figure(figsize=(8,4)); plt.bar([str(i) for i in range(len(sz))], sz)
plt.xlabel('cluster_id'); plt.ylabel('customers'); plt.title('Cluster sizes')
for i, v in enumerate(sz): plt.text(i, v, f'{v}\n({100*v/len(final_labels):.1f}%)', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'cluster_sizes.png', dpi=120); plt.show(); print('saved: cluster_sizes.png')
for f in ['late_payment_ratio','avg_payment_delay','avg_invoice_amount','invoice_frequency','recent_late_ratio']:
    plt.figure(figsize=(8,4)); d = [customer_clusters.loc[customer_clusters['cluster_id']==k, f].dropna().values for k in sorted(set(final_labels))]
    plt.boxplot(d, tick_labels=[f'c{k}' for k in sorted(set(final_labels))])
    plt.title(f'{f} by cluster'); plt.tight_layout(); plt.savefig(OUT_DIR / f'box_{f}.png', dpi=120); plt.show(); print('saved:', f'box_{f}.png')
for k in sorted(set(final_labels)):
    r = prof.loc[prof['cluster_id']==k].iloc[0]
    print(f"Cluster {k}\n---------\nCustomers: {int(r['customer_count'])} ({r['customer_percentage']}%)\nMedian late-payment ratio: {r['median_late_payment_ratio']:.3f}\nMedian avg payment delay: {r['median_avg_payment_delay']:.2f} days\nMedian invoice amount: {r['median_avg_invoice_amount']:.2f}\nBehaviour: observed medians above; descriptive only.")

## 17. Save Clustered Customers

`customer_behavior_clusters.csv` (identifiers + cluster_id + behavioural features); profiles/comparison/stability CSVs. Post-hoc note: cluster late-ratios below reuse clustering features for interpretation — `is_late` never entered the matrix.

In [ ]:
customer_clusters.to_csv(REPO_ROOT / 'data' / 'processed' / 'customer_behavior_clusters.csv', index=False)
print('saved: customer_behavior_clusters.csv', customer_clusters.shape)
posthoc = customer_clusters.groupby('cluster_id')['late_payment_ratio'].median().round(4)
print('Post-hoc cluster late_payment_ratio (interpretation only, feature-derived):'); print(posthoc.to_string())
assert 'is_late' not in CLUSTER_FEATURES

## 18. Save Clustering Model

Fitted estimator + metadata (algorithm/params/K/features/pipeline reference/counts/metrics/stability/seed). K-Means refit with the final seed for a clean saved object.

In [ ]:
if FINAL_ALGORITHM == 'KMeans':
    save_model = KMeans(**FINAL_PARAMETERS); save_model.fit(Xn)
elif FINAL_ALGORITHM.startswith('Agglomerative'):
    save_model = AgglomerativeClustering(n_clusters=FINAL_N_CLUSTERS, linkage=FINAL_PARAMETERS['linkage']); save_model.fit(Xn)
else:
    save_model = DBSCAN(eps=FINAL_PARAMETERS['eps'], min_samples=FINAL_PARAMETERS['min_samples']); save_model.fit(Xn)
joblib.dump({'model': save_model, 'algorithm': FINAL_ALGORITHM, 'parameters': FINAL_PARAMETERS, 'features': CLUSTER_FEATURES}, REPO_ROOT / 'models' / 'customer_behavior_clusterer.joblib')
print('saved: customer_behavior_clusterer.joblib')

In [ ]:
fin = internal_metrics(Xn, final_labels)
stab_df = None
seeds = [0, 21, 42, 84, 123]
aris = []
if FINAL_ALGORITHM == 'KMeans':
    labs = [KMeans(n_clusters=FINAL_N_CLUSTERS, random_state=s, n_init=10).fit_predict(Xn) for s in seeds]
    for i in range(len(seeds)):
        for j in range(i + 1, len(seeds)):
            aris.append(float(adjusted_rand_score(labs[i], labs[j])))
    stab_df = pd.DataFrame({'comparison': [f'{seeds[i]}-vs-{seeds[j]}' for i in range(len(seeds)) for j in range(i+1, len(seeds))], 'ARI': aris})
    stab_df.to_csv(OUT_DIR / 'stability_results.csv', index=False)
    print(stab_df.round(4).to_string(index=False))
    print(f"mean ARI={np.mean(aris):.4f} min={np.min(aris):.4f} max={np.max(aris):.4f} (labels arbitrary -> ARI, not raw label match)")
else:
    print('stability via seeds applies to K-Means; Agglomerative/DBSCAN deterministic for fixed params')
meta = {'algorithm': FINAL_ALGORITHM, 'parameters': FINAL_PARAMETERS, 'number_of_clusters': int(FINAL_N_CLUSTERS),
        'cluster_feature_names': CLUSTER_FEATURES, 'preprocessing_pipeline': str(PIPE_PATH),
        'training_customer_count': int(len(customer_features)), 'silhouette_score': round(float(fin['silhouette']), 4) if fin['silhouette']==fin['silhouette'] else None,
        'calinski_harabasz_score': round(float(fin['ch']), 2) if fin['ch']==fin['ch'] else None,
        'davies_bouldin_score': round(float(fin['db']), 4) if fin['db']==fin['db'] else None,
        'stability_metrics': {'mean_ARI': round(float(np.mean(aris)),4) if aris else None, 'min_ARI': round(float(np.min(aris)),4) if aris else None, 'max_ARI': round(float(np.max(aris)),4) if aris else None, 'seeds': seeds},
        'random_state': RANDOM_STATE}
(REPO_ROOT / 'models' / 'customer_behavior_clustering_metadata.json').write_text(json.dumps(meta, indent=2))
print('saved: customer_behavior_clustering_metadata.json'); print(json.dumps(meta, indent=2))

## 19. Save Evaluation Results

Comparison, profiles and stability CSVs verified on disk.

In [ ]:
import os
need = ['model_comparison.csv','cluster_profiles.csv','stability_results.csv'] if FINAL_ALGORITHM=='KMeans' else ['model_comparison.csv','cluster_profiles.csv']
[print(('OK ' if (OUT_DIR/n).exists() else 'MISSING ') + n) for n in need]
print('clusters file:', (REPO_ROOT/'data'/'processed'/'customer_behavior_clusters.csv').exists())

## 20. Final Clustering Summary

Checklist-gated READY/NOT READY plus the required summary block.

In [ ]:
checks = {'one_row_per_customer': bool(customer_clusters['customer_id'].is_unique and len(customer_clusters)==len(customer_features)),
          'no_id_leak': set(['customer_id','customer_name']).isdisjoint(X.columns) and set(['customer_id','customer_name']).isdisjoint(CLUSTER_FEATURES),
          'no_target': 'is_late' not in CLUSTER_FEATURES, 'no_nan': bool(np.isfinite(Xn).all()),
          'k2': FINAL_N_CLUSTERS >= 2, 'metrics': fin['silhouette']==fin['silhouette'], 'stability': bool(aris) and float(np.mean(aris)) > 0.5 if aris else True,
          'profiles': (OUT_DIR/'cluster_profiles.csv').exists(), 'pca': (OUT_DIR/'pca_clusters.png').exists(),
          'outputs': (REPO_ROOT/'data'/'processed'/'customer_behavior_clusters.csv').exists(),
          'model': (REPO_ROOT/'models'/'customer_behavior_clusterer.joblib').exists(),
          'meta': (REPO_ROOT/'models'/'customer_behavior_clustering_metadata.json').exists()}
ok = all(checks.values())
print('validation:', {k: bool(v) for k, v in checks.items()})
print('========================================================')
print('FLOWCRED AI - CUSTOMER BEHAVIOUR CLUSTERING')
print('========================================================')
print(f'Customers analysed: {len(customer_features)}')
print(f'Features used: {len(CLUSTER_FEATURES)}')
print('--------------------------------------------------------')
print('MODEL COMPARISON')
print('--------------------------------------------------------')
kb = km_df.loc[km_df["silhouette"].idxmax()]; ab = ag_df.loc[ag_df["silhouette"].idxmax()]
print(f"Best evaluated configurations (by silhouette, for display; selection used joint evidence): K-Means: K={int(kb['K'])} Sil={kb['silhouette']:.4f}; Agglomerative: {ab['linkage']} K={int(ab['K'])} Sil={ab['silhouette']:.4f}; DBSCAN: {'none valid' if not db_df['valid'].any() else db_df.loc[db_df['valid'],'eps'].astype(str).str.cat(db_df.loc[db_df['valid'],'min_samples'].astype(str), sep='/').tolist()}")
print('--------------------------------------------------------')
print('FINAL CONFIGURATION')
print('--------------------------------------------------------')
print(f'Algorithm: {FINAL_ALGORITHM}')
print(f'Parameters: {FINAL_PARAMETERS}')
print(f'Number of clusters: {FINAL_N_CLUSTERS}')
print('--------------------------------------------------------')
print('CLUSTER QUALITY')
print('--------------------------------------------------------')
print(f"Silhouette: {fin['silhouette']:.4f}")
print(f"Calinski-Harabasz: {fin['ch']:.2f}")
print(f"Davies-Bouldin: {fin['db']:.4f}")
print('--------------------------------------------------------')
print('STABILITY')
print('--------------------------------------------------------')
print(f"Mean ARI: {float(np.mean(aris)):.4f}" if aris else 'Mean ARI: n/a (non-KMeans final)')
print(f"Minimum ARI: {float(np.min(aris)):.4f}" if aris else 'Minimum ARI: n/a')
print(f"Maximum ARI: {float(np.max(aris)):.4f}" if aris else 'Maximum ARI: n/a')
print('--------------------------------------------------------')
print('CUSTOMER SEGMENTS')
print('--------------------------------------------------------')
for _, r in prof.iterrows():
    print(f"Cluster {int(r['cluster_id'])}:\nCustomers: {int(r['customer_count'])}\nPercentage: {r['customer_percentage']:.2f}%\n")
print('--------------------------------------------------------')
print('OUTPUTS')
print('--------------------------------------------------------')
print(f"Customer clusters: {REPO_ROOT/'data'/'processed'/'customer_behavior_clusters.csv'}")
print(f"Cluster profiles: {OUT_DIR/'cluster_profiles.csv'}")
print(f"Model comparison: {OUT_DIR/'model_comparison.csv'}")
print(f"Stability results: {OUT_DIR/'stability_results.csv'}")
print(f"Clusterer: {REPO_ROOT/'models'/'customer_behavior_clusterer.joblib'}")
print(f"Metadata: {REPO_ROOT/'models'/'customer_behavior_clustering_metadata.json'}")
print(f'Visualizations: {OUT_DIR}')
print('========================================================')
print('CUSTOMER BEHAVIOUR CLUSTERING STATUS:', 'READY' if ok else 'NOT READY')
if not ok: print('FAILED:', [k for k, v in checks.items() if not v])